In [1]:
import os
import gc
import torch
from ultralytics import YOLO

def clear_mps_memory():
    """Apple Silicon(MPS) 메모리 및 가비지 컬렉터를 완전히 비우는 함수"""
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()
    print("\n🧹 [메모리 청소] MPS 캐시 및 가비지 컬렉션 정리가 완료되었습니다.\n")

def run_experiments():
    # 공통 설정
    DATA_PATH = 'data.yaml'
    PROJECT_NAME = 'vehicle_detection_project'
    
    # =========================================================================
    # 🧪 실험 1: 순정 베이스라인 (imgsz=640만 지정)
    # =========================================================================
    print("⏳ [실험 1] 순정 베이스라인 학습 시동 (imgsz=640)")
    model_pure = YOLO('yolov8s.pt')
    
    model_pure.train(
        data=DATA_PATH,
        epochs=300,
        patience=50,          # 50 에폭 동안 성능 개선 없으면 조기 종료
        batch=8,
        imgsz=640,
        device='mps',
        workers=2,
        project=PROJECT_NAME,
        name='baseline_pure', # 결과가 저장될 폴더명
        save=True,
        plots=True
    )
    
    print("📊 [실험 1 평가] 순정 베이스라인 Test 데이터셋 성능 평가 중...")
    # 학습 완료 후 가장 좋았던 최고 가중치(best.pt)로 Test 세트 검증 수행
    pure_test_results = model_pure.val(split='test', project=PROJECT_NAME, name='baseline_pure_test')
    
    # 1번 실험 종료 후 메모리 리셋
    del model_pure
    clear_mps_memory()

    # =========================================================================
    # 🧪 실험 2: 커스텀 고성능 모드 (불균형 방어 및 과적합 차단 옵션)
    # =========================================================================
    print("⏳ [실험 2] 커스텀 고성능 학습 시동 (AdamW + 불균형 방어 옵션)")
    model_custom = YOLO('yolov8s.pt')
    
    model_custom.train(
        data=DATA_PATH,
        epochs=300,
        patience=50,          # 50 에폭 동안 성능 개선 없으면 조기 종료
        batch=8,
        imgsz=640,
        device='mps',
        workers=2,
        project=PROJECT_NAME,
        name='custom_peak',   # 결과가 저장될 폴더명
        save=True,
        plots=True,
        
        # 유저 커스텀 하이퍼파라미터 주입
        optimizer='AdamW',
        lr0=0.001,
        cls=1.5,              # 클래스 불균형 페널티 강화
        mosaic=1.0,           # 모자이크 증강 기본 유지
        mixup=0.2,            # 과적합 방지용 이미지 합성
        degrees=15.0,         # 회전 방어
        hsv_s=0.7,            # 채도 변형
        hsv_v=0.4             # 명도 변형
    )
    
    print("📊 [실험 2 평가] 커스텀 고성능 모드 Test 데이터셋 성능 평가 중...")
    # 학습 완료 후 가장 좋았던 최고 가중치(best.pt)로 Test 세트 검증 수행
    custom_test_results = model_custom.val(split='test', project=PROJECT_NAME, name='custom_peak_test')
    
    # 2번 실험 종료 후 메모리 리셋
    del model_custom
    clear_mps_memory()
    
    print("🎉 모든 실험 및 Test 데이터셋 성능 평가가 성공적으로 종료되었습니다!")
    
    # 간단한 요약 지표 출력
    print("\n=================== [최종 결과 요약] ===================")
    print(f"▶ 순정 베이스라인 - Test mAP50: {pure_test_results.results_dict['metrics/mAP50(B)']:.4f}")
    print(f"▶ 커스텀 고성능  - Test mAP50: {custom_test_results.results_dict['metrics/mAP50(B)']:.4f}")
    print("=======================================================")

if __name__ == '__main__':
    run_experiments()

⏳ [실험 1] 순정 베이스라인 학습 시동 (imgsz=640)
Ultralytics 8.4.62 🚀 Python-3.11.15 torch-2.12.0 MPS (Apple M5)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline_pure, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=

In [1]:
import os
import gc
import torch
from ultralytics import YOLO

def clear_mps_memory():
    """Apple Silicon(MPS) 메모리 및 가비지 컬렉터를 완전히 비우는 함수"""
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()
    print("\n🧹 [메모리 청소] MPS 캐시 및 가비지 컬렉션 정리가 완료되었습니다.\n")

def run_experiments():
    # 공통 설정
    DATA_PATH = 'data.yaml'
    PROJECT_NAME = 'vehicle_detection_project'
    
    # =========================================================================
    # 🧪 실험 1: 순정 베이스라인 (과적합 방지를 위해 150 에폭으로 축소)
    # =========================================================================
    print("⏳ [실험 1] 순정 베이스라인 학습 시동 (imgsz=640)")
    model_pure = YOLO('yolov8s.pt')
    
    model_pure.train(
        data=DATA_PATH,
        epochs=150,           # 데이터 규모에 맞춰 300 -> 150으로 축소
        patience=30,          # 30 에폭 동안 성능 개선 없으면 조기 종료
        batch=8,
        imgsz=640,
        device='mps',
        workers=2,
        project=PROJECT_NAME,
        name='baseline_pure', # 결과가 저장될 폴더명
        save=True,
        plots=True
    )
    
    print("📊 [실험 1 평가] 순정 베이스라인 Test 데이터셋 성능 평가 중...")
    pure_test_results = model_pure.val(split='test', project=PROJECT_NAME, name='baseline_pure_test')
    
    # 1번 실험 종료 후 메모리 리셋
    del model_pure
    clear_mps_memory()

    # =========================================================================
    # 🧪 실험 2: 커스텀 전이 학습 모드 (백본 프리즈 + 불균형 방어)
    # =========================================================================
    print("⏳ [실험 2] 시각 세포 얼리기(Freeze) 및 커스텀 전이 학습 시동")
    model_custom = YOLO('yolov8s.pt')
    
    model_custom.train(
        data=DATA_PATH,
        epochs=150,           # 데이터 규모에 맞춰 300 -> 150으로 축소
        patience=30,          # 30 에폭 동안 성능 개선 없으면 조기 종료
        batch=8,
        imgsz=640,
        device='mps',
        workers=2,
        project=PROJECT_NAME,
        name='custom_frozen', # 기존 custom_peak에서 이름 변경
        save=True,
        plots=True,
        
        # 🔥 파멸적 망각 방지 핵심 옵션
        freeze=10,            # 0~9번 레이어(백본) 가중치 잠금 (기존 지식 보호)
        optimizer='AdamW',
        lr0=0.001,            # 가중치를 얼려두었으므로 안정적인 속도로 튜닝 가능
        
        # 유저 커스텀 하이퍼파라미터 (소수 클래스 방어)
        cls=1.5,              # 클래스 불균형 페널티 강화
        mosaic=1.0,           
        mixup=0.1,            # 데이터가 적으므로 너무 강한 증강은 독이 됨 (0.2 -> 0.1 완화)
        degrees=15.0,         
        hsv_s=0.7,            
        hsv_v=0.4             
    )
    
    print("📊 [실험 2 평가] 커스텀 전이 학습 모드 Test 데이터셋 성능 평가 중...")
    custom_test_results = model_custom.val(split='test', project=PROJECT_NAME, name='custom_frozen_test')
    
    # 2번 실험 종료 후 메모리 리셋
    del model_custom
    clear_mps_memory()
    
    print("🎉 모든 실험 및 Test 데이터셋 성능 평가가 성공적으로 종료되었습니다!")
    
    # 간단한 요약 지표 출력
    print("\n=================== [최종 결과 요약] ===================")
    print(f"▶ 순정 베이스라인 - Test mAP50: {pure_test_results.results_dict['metrics/mAP50(B)']:.4f}")
    print(f"▶ 커스텀 전이학습 - Test mAP50: {custom_test_results.results_dict['metrics/mAP50(B)']:.4f}")
    print("=======================================================")

if __name__ == '__main__':
    run_experiments()

⏳ [실험 1] 순정 베이스라인 학습 시동 (imgsz=640)
New https://pypi.org/project/ultralytics/8.4.63 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.62 🚀 Python-3.11.15 torch-2.12.0 MPS (Apple M5)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=basel

In [1]:
import os
import gc
import torch
from ultralytics import YOLO

def clear_mps_memory():
    """Apple Silicon(MPS) 메모리 및 가비지 컬렉터를 완전히 비우는 함수"""
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()
    print("\n🧹 [메모리 청소] MPS 캐시 및 가비지 컬렉션 정리가 완료되었습니다.\n")

def run_experiments():
    # 공통 설정
    DATA_PATH = 'data.yaml'
    PROJECT_NAME = 'vehicle_detection_project'
    
    # =========================================================================
    # 🧪 실험 1: 순정 베이스라인 (시간 절약을 위해 50 에폭으로 축소)
    # =========================================================================
    print("⏳ [실험 1] 순정 베이스라인 학습 시동 (yolov8s, imgsz=640)")
    model_pure = YOLO('yolov8s.pt')
    
    model_pure.train(
        data=DATA_PATH,
        epochs=50,            # 딱 50까지만 열어둠
        patience=20,          # 20 에폭 동안 성능 개선 없으면 조기 종료
        batch=8,
        imgsz=640,
        device='mps',
        workers=2,
        project=PROJECT_NAME,
        name='baseline_pure_50', 
        save=True,
        plots=True
    )
    
    print("📊 [실험 1 평가] 순정 베이스라인 Test 데이터셋 성능 평가 중...")
    pure_test_results = model_pure.val(split='test', project=PROJECT_NAME, name='baseline_pure_50_test')
    
    # 1번 실험 종료 후 메모리 리셋
    del model_pure
    clear_mps_memory()

    # =========================================================================
    # 🧪 실험 2: 커스텀 전이 학습 모드 (백본 프리즈 + 불균형 방어)
    # =========================================================================
    print("⏳ [실험 2] 시각 세포 얼리기(Freeze) 및 커스텀 전이 학습 시동")
    model_custom = YOLO('yolov8s.pt')
    
    model_custom.train(
        data=DATA_PATH,
        epochs=50,            # 딱 50까지만 열어둠
        patience=20,          # 20 에폭 동안 성능 개선 없으면 조기 종료
        batch=8,
        imgsz=640,
        device='mps',
        workers=2,
        project=PROJECT_NAME,
        name='custom_frozen_50', 
        save=True,
        plots=True,
        
        # 🔥 파멸적 망각 방지 핵심 옵션
        freeze=10,            # 0~9번 레이어(백본) 가중치 잠금
        optimizer='AdamW',
        lr0=0.001,            
        
        # 유저 커스텀 하이퍼파라미터 (소수 클래스 방어)
        cls=1.5,              
        mosaic=1.0,           
        mixup=0.1,            
        degrees=15.0,         
        hsv_s=0.7,            
        hsv_v=0.4             
    )
    
    print("📊 [실험 2 평가] 커스텀 전이 학습 모드 Test 데이터셋 성능 평가 중...")
    custom_test_results = model_custom.val(split='test', project=PROJECT_NAME, name='custom_frozen_50_test')
    
    # 2번 실험 종료 후 메모리 리셋
    del model_custom
    clear_mps_memory()
    
    print("🎉 모든 실험 및 Test 데이터셋 성능 평가가 성공적으로 종료되었습니다!")
    
    # 간단한 요약 지표 출력
    print("\n=================== [최종 결과 요약] ===================")
    print(f"▶ 순정 베이스라인(50e) - Test mAP50: {pure_test_results.results_dict['metrics/mAP50(B)']:.4f}")
    print(f"▶ 커스텀 전이학습(50e) - Test mAP50: {custom_test_results.results_dict['metrics/mAP50(B)']:.4f}")
    print("=======================================================")

if __name__ == '__main__':
    run_experiments()

⏳ [실험 1] 순정 베이스라인 학습 시동 (yolov8s, imgsz=640)
New https://pypi.org/project/ultralytics/8.4.64 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.62 🚀 Python-3.11.15 torch-2.12.0 MPS (Apple M5)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, na

In [2]:
import os
import gc
import torch
from ultralytics import YOLO

def clear_mps_memory():
    """Apple Silicon(MPS) 메모리 및 가비지 컬렉터를 완전히 비우는 함수"""
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()
    print("\n🧹 [메모리 청소] MPS 캐시 및 가비지 컬렉션 정리가 완료되었습니다.\n")

def run_experiments():
    # 공통 설정
    DATA_PATH = 'data.yaml'
    PROJECT_NAME = 'vehicle_detection_project'
    
    # =========================================================================
    # 🧪 실험 1: 순정 베이스라인 (30 에폭)
    # =========================================================================
    print("⏳ [실험 1] 순정 베이스라인 학습 시동 (yolov8s, imgsz=640, epochs=30)")
    model_pure = YOLO('yolov8s.pt')
    
    model_pure.train(
        data=DATA_PATH,
        epochs=30,            # 딱 30까지만 열어둠
        patience=15,          # 15 에폭 동안 성능 개선 없으면 조기 종료
        batch=8,
        imgsz=640,
        device='mps',
        workers=2,
        project=PROJECT_NAME,
        name='baseline_pure_30', 
        save=True,
        plots=True
    )
    
    print("📊 [실험 1 평가] 순정 베이스라인 Test 데이터셋 성능 평가 중...")
    pure_test_results = model_pure.val(split='test', project=PROJECT_NAME, name='baseline_pure_30_test')
    
    # 1번 실험 종료 후 메모리 리셋
    del model_pure
    clear_mps_memory()

    # =========================================================================
    # 🧪 실험 2: 커스텀 전이 학습 모드 (백본 프리즈 + 30 에폭)
    # =========================================================================
    print("⏳ [실험 2] 시각 세포 얼리기(Freeze) 및 커스텀 전이 학습 시동")
    model_custom = YOLO('yolov8s.pt')
    
    model_custom.train(
        data=DATA_PATH,
        epochs=30,            # 딱 30까지만 열어둠
        patience=15,          # 15 에폭 동안 성능 개선 없으면 조기 종료
        batch=8,
        imgsz=640,
        device='mps',
        workers=2,
        project=PROJECT_NAME,
        name='custom_frozen_30', 
        save=True,
        plots=True,
        
        # 🔥 파멸적 망각 방지 핵심 옵션
        freeze=10,            # 0~9번 레이어(백본) 가중치 잠금
        optimizer='AdamW',
        lr0=0.001,            
        
        # 유저 커스텀 하이퍼파라미터 (소수 클래스 방어)
        cls=1.5,              
        mosaic=1.0,           
        mixup=0.1,            
        degrees=15.0,         
        hsv_s=0.7,            
        hsv_v=0.4             
    )
    
    print("📊 [실험 2 평가] 커스텀 전이 학습 모드 Test 데이터셋 성능 평가 중...")
    custom_test_results = model_custom.val(split='test', project=PROJECT_NAME, name='custom_frozen_30_test')
    
    # 2번 실험 종료 후 메모리 리셋
    del model_custom
    clear_mps_memory()
    
    print("🎉 모든 실험 및 Test 데이터셋 성능 평가가 성공적으로 종료되었습니다!")
    
    # 간단한 요약 지표 출력
    print("\n=================== [최종 결과 요약] ===================")
    print(f"▶ 순정 베이스라인(30e) - Test mAP50: {pure_test_results.results_dict['metrics/mAP50(B)']:.4f}")
    print(f"▶ 커스텀 전이학습(30e) - Test mAP50: {custom_test_results.results_dict['metrics/mAP50(B)']:.4f}")
    print("=======================================================")

if __name__ == '__main__':
    run_experiments()

⏳ [실험 1] 순정 베이스라인 학습 시동 (yolov8s, imgsz=640, epochs=30)
New https://pypi.org/project/ultralytics/8.4.64 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.62 🚀 Python-3.11.15 torch-2.12.0 MPS (Apple M5)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_sc

In [2]:
import os
import cv2
import yaml
from collections import defaultdict, deque, Counter
from ultralytics import YOLO

# 1. 깐깐함을 덜어내고 기억력은 코끼리 급으로 올린 트래커 설정
tracker_config = {
    'tracker_type': 'bytetrack',
    'track_buffer': 300,           # [수정] 100 -> 300 (약 10초간 놓쳐도 기억 유지!)
    'track_high_thresh': 0.3,      
    'track_low_thresh': 0.1,
    'new_track_thresh': 0.3,       
    'match_thresh': 0.8,
    'fuse_score': True,
    'gmc_method': 'sparseOptFlow'
}
with open('my_tracker.yaml', 'w') as f:
    yaml.dump(tracker_config, f)

# ==========================================
# 2. 모델 로드 및 매핑 딕셔너리 설정
# ==========================================
model = YOLO('./my_output_weights/vehicle_yolov8s_focus.pt') 

# 파편화된 클래스 ID를 큰 범주로 묶어주는 규칙
CLASS_MAPPING = {
    0: 'Bus', 2: 'Bus', 3: 'Bus', 6: 'Bus',               # 버스류 4개
    1: 'Truck', 5: 'Truck', 7: 'Truck', 8: 'Truck',       # 트럭류 7개
    9: 'Truck', 10: 'Truck', 11: 'Truck',
    4: 'Car'                                              # 승용차
}

# 시각화 색상표 (BGR 기준)
COLOR_MAPPING = {
    'Bus': (255, 100, 100),   # 파란색 계열
    'Truck': (100, 100, 255), # 빨간색 계열
    'Car': (100, 255, 100)    # 초록색 계열
}

# ==========================================
# 3. 비디오 입출력 설정 (MP4 포맷)
# ==========================================
video_path = 'cctv_video.mp4'
output_path = 'demo_final_mapped.mp4' 
class_history = defaultdict(lambda: deque(maxlen=20))

cap = cv2.VideoCapture(video_path)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# MP4 저장을 위한 코덱 설정
out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

print("🎥 최종 시연 영상을 렌더링합니다. (설정: 깐깐함 완화, MP4 출력, 로그 정제)")

# ==========================================
# 4. 모델 추론 및 후처리 루프
# ==========================================
results = model.track(
    source=video_path, 
    stream=True, 
    tracker='my_tracker.yaml', 
    conf=0.15,           # YOLO 기본 커트라인 완화 (0.25 -> 0.15)
    imgsz=960, 
    persist=True, 
    device='mps',
    verbose=False        # YOLO의 시끄러운 기본 로그 비활성화
)

for frame_idx, r in enumerate(results):
    frame = r.orig_img.copy()
    boxes = r.boxes
    
    current_frame_objects = [] # 터미널 출력을 위한 현재 프레임 차량 리스트
    
    if boxes.id is not None:
        ids = boxes.id.int().cpu().tolist()
        cls = boxes.cls.int().cpu().tolist()
        xyxy = boxes.xyxy.cpu().numpy()
        
        for i, box in enumerate(xyxy):
            track_id = ids[i]
            original_cls_id = cls[i]
            
            # 1. 큰 범주로 클래스 이름 변환
            mapped_cls_name = CLASS_MAPPING.get(original_cls_id, 'Unknown')
            
            # 2. 깜빡임 방지를 위한 다수결 투표
            class_history[track_id].append(mapped_cls_name)
            final_cls_name = Counter(class_history[track_id]).most_common(1)[0][0]
            
            current_frame_objects.append(final_cls_name)
            
            # 3. 화면에 박스 및 텍스트 그리기
            x1, y1, x2, y2 = map(int, box)
            color = COLOR_MAPPING.get(final_cls_name, (255, 255, 255))
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            
            label = f"ID:{track_id} {final_cls_name}"
            cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
            
    # 정제된 진행 상황 터미널 출력
    print(f"🎬 프레임 [{frame_idx:04d}]: 감지된 차량 -> {dict(Counter(current_frame_objects))}")
            
    out.write(frame)

cap.release()
out.release()
print(f"\n🎉 모든 작업 완료! 렌더링 된 파일 위치: {os.path.abspath(output_path)}")

🎥 최종 시연 영상을 렌더링합니다. (설정: 깐깐함 완화, MP4 출력, 로그 정제)
🎬 프레임 [0000]: 감지된 차량 -> {'Car': 12}
🎬 프레임 [0001]: 감지된 차량 -> {'Car': 12}
🎬 프레임 [0002]: 감지된 차량 -> {'Car': 12}
🎬 프레임 [0003]: 감지된 차량 -> {'Car': 12}
🎬 프레임 [0004]: 감지된 차량 -> {'Car': 12}
🎬 프레임 [0005]: 감지된 차량 -> {'Car': 12}
🎬 프레임 [0006]: 감지된 차량 -> {'Car': 12}
🎬 프레임 [0007]: 감지된 차량 -> {'Car': 12}
🎬 프레임 [0008]: 감지된 차량 -> {'Car': 11}
🎬 프레임 [0009]: 감지된 차량 -> {'Car': 11}
🎬 프레임 [0010]: 감지된 차량 -> {'Car': 11}
🎬 프레임 [0011]: 감지된 차량 -> {'Car': 12}
🎬 프레임 [0012]: 감지된 차량 -> {'Car': 10}
🎬 프레임 [0013]: 감지된 차량 -> {'Car': 10}
🎬 프레임 [0014]: 감지된 차량 -> {'Car': 11}
🎬 프레임 [0015]: 감지된 차량 -> {'Car': 13}
🎬 프레임 [0016]: 감지된 차량 -> {'Car': 12}
🎬 프레임 [0017]: 감지된 차량 -> {'Car': 12}
🎬 프레임 [0018]: 감지된 차량 -> {'Car': 10}
🎬 프레임 [0019]: 감지된 차량 -> {'Car': 10}
🎬 프레임 [0020]: 감지된 차량 -> {'Car': 11}
🎬 프레임 [0021]: 감지된 차량 -> {'Car': 12}
🎬 프레임 [0022]: 감지된 차량 -> {'Car': 11}
🎬 프레임 [0023]: 감지된 차량 -> {'Car': 12}
🎬 프레임 [0024]: 감지된 차량 -> {'Car': 10}
🎬 프레임 [0025]: 감지된 차량 -> {'Car': 11}
🎬 프레임 [0026]: 감지

In [5]:
import os
from ultralytics import YOLO

# 1. 승리한 최적화 모델 가중치 파일 불러오기
model_path = './my_output_weights/vehicle_yolov8s_optimized.pt'
model = YOLO(model_path)

# 2. 알려주신 Test 데이터셋 절대 경로 설정
test_dir = '/Users/gyuminkang/Desktop/YOLO/test'
test_images_dir = os.path.join(test_dir, 'images') # 시연 이미지를 뽑을 폴더

# 3. Test 데이터 성적표(Metrics) 평가
print("📊 Test 데이터셋 기준 최종 성능 평가를 시작합니다...")
# data.yaml 안에 정의된 test 경로의 이미지와 라벨(정답지)을 비교해 채점합니다.
metrics = model.val(data='data.yaml', split='test', device='mps')

# 깔끔한 최종 리포트 출력
print("\n" + "="*60)
print("🏆 [Test 데이터(실전) 기준 최종 성능 성적표] 🏆")
print("="*60)
print(f" 1. Precision (정밀도)     : {metrics.box.mp:.4f}")
print(f" 2. Recall (재현율)        : {metrics.box.mr:.4f}")
print(f" 3. mAP50 (기준 정확도)    : {metrics.box.map50:.4f}")
print(f" 4. mAP50-95 (정밀 정확도) : {metrics.box.map:.4f}")
print("="*60)

# 4. Test 데이터 이미지들에 직접 네모 박스 시각화 및 저장
print("\n📸 알려주신 경로의 Test 이미지들에 차량 인식 박스를 그리고 저장합니다...")
results = model.predict(
    source=test_images_dir,  # [핵심] 알려주신 절대 경로의 images 폴더 지정
    save=True,               # 예측된 박스가 그려진 이미지를 파일로 저장
    conf=0.5,               # 신뢰도 25% 이상만 표시
    device='mps'             # 맥북 가속
)

print("\n" + "="*60)
print("🎉 모든 Test 데이터 시연 이미지 생성이 완료되었습니다!")
# results[0].save_dir는 결과물이 저장된 최종 폴더 경로를 자동으로 찾아줍니다.
print(f"📍 결과물(사진) 확인 경로: {results[0].save_dir}")
print("="*60)

📊 Test 데이터셋 기준 최종 성능 평가를 시작합니다...
Ultralytics 8.4.60 🚀 Python-3.11.15 torch-2.12.0 MPS (Apple M5)
Model summary (fused): 73 layers, 11,130,228 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 196.6±35.8 MB/s, size: 45.9 KB)
val: Scanning /Users/gyuminkang/Desktop/YOLO/test/labels.cache... 458 images, 4 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 458/458 40.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 3.8it/s 7.6s0.3s
                   all        458       6222      0.501      0.578      0.428      0.283
               big bus         82        110       0.84      0.431      0.739        0.5
             big truck        240        648      0.742      0.469      0.562      0.343
                bus-s-         15         16      0.402      0.625      0.442      0.317
                   car        445       4021      0.836        0.7      0.766       0.43
             mid 

In [6]:
import os
from ultralytics import YOLO

# 1. 승리한 최적화 모델 가중치 파일 불러오기
model_path = './my_output_weights/vehicle_yolov8s_optimized.pt'
model = YOLO(model_path)

# 2. 가져오신 CCTV 동영상 절대 경로 설정
video_path = '/Users/gyuminkang/Desktop/YOLO/cctv_video.mp4'

# 3. 모델을 동영상에 주입하여 추론(바운딩 박스 그리기) 시작
print("🎥 고속도로 CCTV 영상 분석을 시작합니다. 렌더링에 시간이 조금 걸릴 수 있습니다...")

results = model.predict(
    source=video_path,
    save=True,           # 바운딩 박스가 쳐진 결과 영상을 파일로 저장
    
    # --- 고속도로 CCTV 맞춤형 핵심 세팅 ---
    imgsz=1280,          # [핵심] 원거리의 좁쌀만 한 차량도 잡기 위해 해상도를 2배(1280)로 확장
    conf=0.35,           # [핵심] 35% 이상 확실할 때만 박스를 쳐서 글씨/화살표 오검출 방지
    
    device='mps'         # 맥북 Apple Silicon GPU로 초고속 렌더링 가속
)

print("\n" + "="*60)
print("🎉 실전 동영상 렌더링이 완벽하게 끝났습니다!")
# 동영상 결과물이 저장된 폴더 경로를 출력해 줍니다.
print(f"📍 결과 영상 확인 경로: {results[0].save_dir}")
print("="*60)

🎥 고속도로 CCTV 영상 분석을 시작합니다. 렌더링에 시간이 조금 걸릴 수 있습니다...

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/678) /Users/gyuminkang/Desktop/YOLO/cctv_video.mp4: 832x1280 22 cars, 2285.3ms
video 1/1 (frame 2/678) /Users/gyuminkang/Desktop/YOLO/cctv_video.mp4: 832x1280 22 cars, 24.1ms
video 1/1 (frame 3/678) /Users/gyuminkang/Desktop/YOLO/cctv_video.mp4: 832x1280 22 cars, 36.8ms
video 1/1 (frame 4/678) /Users/gyuminkang/Desktop/YOLO/cctv_video.mp4: 832x1280 23 cars, 21.4ms
video 1/1 (frame 5/67

In [7]:
import os
from ultralytics import YOLO

# 1. 승리한 최적화 모델 가중치 파일 불러오기
model_path = './my_output_weights/vehicle_yolov8s_optimized.pt'
model = YOLO(model_path)

# 2. CCTV 동영상 절대 경로
video_path = '/Users/gyuminkang/Desktop/YOLO/cctv_video.mp4'

# 3. [핵심] predict가 아닌 'track'을 사용하여 동영상 렌더링
print("🎥 ByteTrack 알고리즘을 적용하여 부드러운 객체 추적 영상을 생성합니다...")

results = model.track(
    source=video_path,
    save=True,           
    imgsz=1280,          
    conf=0.35,           
    
    # --- 깐깐한 추적(Tracking)을 위한 핵심 세팅 ---
    persist=True,               # [핵심] 객체에 ID(번호표)를 부여하고 계속 기억함
    tracker="bytetrack.yaml",   # [핵심] 겹치는 차량, 깜빡이는 박스를 강력하게 보정해주는 알고리즘
    
    device='mps'         
)

print("\n" + "="*60)
print("🎉 부드러운 Tracking 동영상 렌더링이 완료되었습니다!")
print(f"📍 결과 영상 확인 경로: {results[0].save_dir}")
print("="*60)

🎥 ByteTrack 알고리즘을 적용하여 부드러운 객체 추적 영상을 생성합니다...
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.5 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 34.8 MB/s  0:00:00

requirements: AutoUpdate success ✅ 0.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/678) /Users/gyuminkang/Desktop/Y

In [1]:
import os
from ultralytics import YOLO

model = YOLO('./my_output_weights/vehicle_yolov8s_optimized.pt')
video_path = '/Users/gyuminkang/Desktop/YOLO/cctv_video.mp4'

print("🎥 ByteTrack 파라미터 튜닝 모드로 렌더링을 시작합니다...")

results = model.track(
    source=video_path,
    save=True,           
    imgsz=1280,          
    
    # --- 깜빡임(Flickering) 제거를 위한 핵심 튜닝 ---
    conf=0.1,           # [핵심 1] 컷오프를 10%로 대폭 낮춤 (ByteTrack에 충분한 재료 공급)
    iou=0.5,            # [핵심 2] 박스들이 과도하게 겹치는 것을 정리 (NMS 임계값 조절)
    
    persist=True,               
    tracker="bytetrack.yaml",   
    device='mps'         
)

print(f"\n🎉 튜닝된 Tracking 렌더링 완료! 경로: {results[0].save_dir}")

🎥 ByteTrack 파라미터 튜닝 모드로 렌더링을 시작합니다...

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/678) /Users/gyuminkang/Desktop/YOLO/cctv_video.mp4: 832x1280 27 cars, 89.8ms
video 1/1 (frame 2/678) /Users/gyuminkang/Desktop/YOLO/cctv_video.mp4: 832x1280 26 cars, 18.9ms
video 1/1 (frame 3/678) /Users/gyuminkang/Desktop/YOLO/cctv_video.mp4: 832x1280 26 cars, 17.2ms
video 1/1 (frame 4/678) /Users/gyuminkang/Desktop/YOLO/cctv_video.mp4: 832x1280 27 cars, 17.1ms
video 1/1 (frame 5/678) /Users/gyumi

In [11]:
import yaml
from ultralytics import YOLO

# [핵심 수정] 에러 방지를 위해 필요한 모든 설정을 다 집어넣습니다.
tracker_config = {
    'tracker_type': 'bytetrack',
    'track_buffer': 100,         # 기억력 강화
    'track_high_thresh': 0.5,
    'track_low_thresh': 0.1,
    'new_track_thresh': 0.6,
    'match_thresh': 0.8,
    'fuse_score': True,          # 에러 해결 핵심: 이 값이 없어서 에러가 났었습니다
    'gmc_method': 'sparseOptFlow'
}

# 설정 파일 저장
with open('my_tracker.yaml', 'w') as f:
    yaml.dump(tracker_config, f)

# 모델 로드
model = YOLO('./my_output_weights/vehicle_yolov8s_optimized.pt')
video_path = '/Users/gyuminkang/Desktop/YOLO/cctv_video.mp4'

print("🎥 시스템 업데이트 및 설정 보강 완료. 다시 추적을 시작합니다...")

results = model.track(
    source=video_path,
    save=True,           
    imgsz=1280,          
    conf=0.1,            
    iou=0.5,             
    persist=True,               
    tracker='my_tracker.yaml', # 방금 보강된 설정 파일 사용
    device='mps'         
)

print(f"\n🎉 렌더링 완료! 결과 경로: {results[0].save_dir}")

🎥 시스템 업데이트 및 설정 보강 완료. 다시 추적을 시작합니다...

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/678) /Users/gyuminkang/Desktop/YOLO/cctv_video.mp4: 832x1280 12 cars, 26.8ms
video 1/1 (frame 2/678) /Users/gyuminkang/Desktop/YOLO/cctv_video.mp4: 832x1280 12 cars, 18.0ms
video 1/1 (frame 3/678) /Users/gyuminkang/Desktop/YOLO/cctv_video.mp4: 832x1280 12 cars, 16.8ms
video 1/1 (frame 4/678) /Users/gyuminkang/Desktop/YOLO/cctv_video.mp4: 832x1280 13 cars, 17.0ms
video 1/1 (frame 5/678) /Users/gyum

In [13]:
import os
import cv2
import yaml
from collections import defaultdict, deque, Counter
from ultralytics import YOLO
from ultralytics.utils.plotting import Annotator, colors

# 1. 환경 설정 및 커스텀 트래커 파일 생성
# 시스템의 호환성 문제를 방지하기 위해 필요한 설정을 모두 포함합니다.
os.chdir('/Users/gyuminkang/Desktop/YOLO/')
tracker_config = {
    'tracker_type': 'bytetrack',
    'track_buffer': 100,           # 객체 기억력 강화 (깜빡임 방지)
    'track_high_thresh': 0.5,
    'track_low_thresh': 0.1,
    'new_track_thresh': 0.6,
    'match_thresh': 0.8,
    'fuse_score': True,
    'gmc_method': 'sparseOptFlow'
}
with open('my_tracker.yaml', 'w') as f:
    yaml.dump(tracker_config, f)

# 2. 모델 및 영상 준비
model = YOLO('./my_output_weights/vehicle_yolov8s_optimized.pt')
video_path = 'cctv_video.mp4'
output_path = 'final_output_fixed.mp4'

# 클래스 이력 저장소 (최근 20프레임 중 가장 많이 나온 클래스 선정)
class_history = defaultdict(lambda: deque(maxlen=20))

cap = cv2.VideoCapture(video_path)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# 비디오 저장 설정
out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

print("🎥 클래스 고정 및 궤적 추적 영상 렌더링을 시작합니다...")

# 3. 추적 및 보정 루프
# stream=True로 설정하여 메모리 효율을 극대화합니다.
results = model.track(
    source=video_path, 
    stream=True, 
    tracker='my_tracker.yaml', 
    conf=0.1, 
    imgsz=1280, 
    persist=True,
    device='mps'
)

for r in results:
    annotator = Annotator(r.orig_img)
    boxes = r.boxes
    
    if boxes.id is not None:
        ids = boxes.id.int().cpu().tolist()
        cls = boxes.cls.int().cpu().tolist()
        xyxy = boxes.xyxy.cpu().numpy()
        
        for i, box in enumerate(xyxy):
            track_id = ids[i]
            current_cls = cls[i]
            
            # [핵심] 클래스 투표 로직 (최근 20개 중 최빈값 선택)
            class_history[track_id].append(current_cls)
            most_frequent_cls = Counter(class_history[track_id]).most_common(1)[0][0]
            
            # [핵심] 줏대 있는 라벨링
            label = f"ID:{track_id} {r.names[most_frequent_cls]}"
            
            # [핵심] OpenCV 에러 방지용 컬러 튜플 생성 (BGR 방식)
            color = colors(most_frequent_cls, bgr=True)
            
            # 박스 및 라벨 그리기
            annotator.box_label(box, label=label, color=color)
    
    # 결과 영상 쓰기
    frame = annotator.result()
    out.write(frame)

cap.release()
out.release()

print(f"\n🎉 렌더링 완료! 결과물 위치: {os.path.join(os.getcwd(), output_path)}")

🎥 클래스 고정 및 궤적 추적 영상 렌더링을 시작합니다...

video 1/1 (frame 1/678) /Users/gyuminkang/Desktop/YOLO/cctv_video.mp4: 832x1280 12 cars, 31.6ms
video 1/1 (frame 2/678) /Users/gyuminkang/Desktop/YOLO/cctv_video.mp4: 832x1280 12 cars, 17.4ms
video 1/1 (frame 3/678) /Users/gyuminkang/Desktop/YOLO/cctv_video.mp4: 832x1280 12 cars, 17.5ms
video 1/1 (frame 4/678) /Users/gyuminkang/Desktop/YOLO/cctv_video.mp4: 832x1280 13 cars, 18.8ms
video 1/1 (frame 5/678) /Users/gyuminkang/Desktop/YOLO/cctv_video.mp4: 832x1280 13 cars, 18.4ms
video 1/1 (frame 6/678) /Users/gyuminkang/Desktop/YOLO/cctv_video.mp4: 832x1280 13 cars, 19.5ms
video 1/1 (frame 7/678) /Users/gyuminkang/Desktop/YOLO/cctv_video.mp4: 832x1280 13 cars, 18.4ms
video 1/1 (frame 8/678) /Users/gyuminkang/Desktop/YOLO/cctv_video.mp4: 832x1280 13 cars, 20.3ms
video 1/1 (frame 9/678) /Users/gyuminkang/Desktop/YOLO/cctv_video.mp4: 832x1280 13 cars, 20.7ms
video 1/1 (frame 10/678) /Users/gyuminkang/Desktop/YOLO/cctv_video.mp4: 832x1280 13 cars, 20.3ms
vide